### Keras Tuner - Hyperparameter Tuning

Simple example of hyperparameter tuning using the Keras Tuner with TensorFlow. The example demonstrates tuning the number of units in a dense layer and the learning rate of the optimizer for a neural network:

In [5]:
import tensorflow as tf
from tensorflow import keras
from kerastuner import HyperModel, RandomSearch

In [2]:
tf.__version__

'2.19.1'

In [6]:
# define a model-builiding function that accepts hyperparameters
def build_model(hp):
    model = keras.Sequential()
    model.add(keras.layers.Flatten(input_shape=(28, 28)))

    # tune number of units in the first dense layer between 32 and 512
    model.add(keras.layers.Dense(units=hp.Int("units", min_value=32, max_value=512, step=32), activation="relu"))

    model.add(keras.layers.Dense(10, activation="softmax"))

    # tune learning rate for the optimizer
    model.compile(
        optimizer=keras.optimizers.Adam(hp.Float("learning_rate", min_value=1e-4, max_value=1e-2, sampling="LOG")),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# instantiate the tuner
tuner = RandomSearch(
    build_model,
    objective="val_accuracy",
    max_trials=5,
    executions_per_trial=3,
    directory="tuner",
    project_name="keras-tuner-mnist-fashion"
)

# load data
(x_train, y_train), (x_val, y_val) = keras.datasets.fashion_mnist.load_data()

# perform hyperparameter search
tuner.search(x_train, y_train, epochs=5, validation_data=(x_val, y_val))

Trial 5 Complete [00h 01m 11s]
val_accuracy: 0.742900013923645

Best val_accuracy So Far: 0.8411999940872192
Total elapsed time: 00h 08m 59s


In [4]:
# get the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"Best number of units: {best_hps.get('units')}")
print(f"Best learning rate: {best_hps.get('learning_rate')}")

Best number of units: 448
Best learning rate: 0.001308287664940752
